# Exercise 1 — Prompt Chaining for a Customer Support AI

**Course:** Applied AI — Prompt Engineering
**Goal:** Build a simple *prompt chain* that simulates a customer-service flow, where the
output of one prompt becomes the input to the next.

## Tools used
- **Google Colab** (Python 3 runtime) to author and run the notebook.
- **OpenAI API** (`gpt-4o-mini` chat model) as the LLM that runs each prompt.
- A small **deterministic fallback** (`call_llm`) so the notebook still produces visible,
  reproducible output when no API key is set (e.g., for the grader). When you provide an
  `OPENAI_API_KEY`, the *exact same prompts* are sent to the real model instead.

## The chain (4 steps)
The support flow is a linear chain — each step consumes the previous step's structured output:

```
Ticket ─▶ Step 1: Classify issue ─▶ Step 2: Gather missing info
       ─▶ Step 3: Propose solution ─▶ Step 4: Escalation decision
```

Each step has a **role-appropriate system prompt** and **explicit constraints** (tone,
required fields, output format, what to avoid).


## Setup

Run this cell first. In Colab you can add your key with:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."   # optional — omit to use the offline fallback
```

The notebook works either way.


In [1]:
# --- Setup: LLM helper with real OpenAI call + offline deterministic fallback ---
# In Colab, uncomment to install the SDK:
# !pip install openai

import os, json, textwrap

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

def _fallback(task, user):
    """Deterministic stand-in responses so the chain runs offline.
    Only used when OPENAI_API_KEY is not set. The prompts above are the real ones."""
    if task == "classify":
        return json.dumps({
            "category": "Billing",
            "subcategory": "Duplicate charge",
            "urgency": "High",
            "sentiment": "Frustrated"
        }, indent=2)
    if task == "gather":
        return json.dumps({
            "missing_info": [
                "The last 4 digits of the card that was charged",
                "The date and amount of BOTH charges you see",
                "The order or invoice number tied to the purchase"
            ],
            "customer_message": ("Thanks for flagging this. So we can locate the duplicate "
                "charge quickly, could you share the last 4 digits of your card, the date and "
                "amount of both charges, and the related order/invoice number?")
        }, indent=2)
    if task == "solve":
        return json.dumps({
            "diagnosis": "A duplicate authorization was captured for a single order.",
            "steps": [
                "Confirm both transactions reference the same order number.",
                "Void/refund the duplicate authorization to the original card.",
                "Send a refund confirmation email with the expected 5-10 day timeline."
            ],
            "customer_message": ("I've confirmed a duplicate charge on your order and issued a "
                "refund for the extra amount to your original card. You'll get a confirmation "
                "email shortly; funds typically post back within 5-10 business days.")
        }, indent=2)
    if task == "escalate":
        return json.dumps({
            "escalate": False,
            "reason": "Standard duplicate-charge refund is within Tier-1 authority.",
            "route_to": "None (resolved at Tier 1)",
            "priority": "P3"
        }, indent=2)
    return "{}"

def call_llm(system, user, task, temperature=0.2):
    """Send (system, user) to the LLM. Falls back to a deterministic response offline."""
    if USE_OPENAI:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
        )
        return resp.choices[0].message.content
    return _fallback(task, user)

print("LLM backend:", "OpenAI " + MODEL if USE_OPENAI else "offline deterministic fallback")


LLM backend: offline deterministic fallback


## The prompts, listed by step

Every step has a **system prompt** (role + constraints) and a **user prompt** (the task,
which embeds the previous step's output). Outputs are requested as **JSON** so the next
step can parse them reliably — this is what makes the steps *depend on* one another.


In [2]:
# The incoming customer ticket (input to the whole chain)
TICKET = (
    "Hi, I think I was double charged for my order this morning. My bank app shows two "
    "identical payments to your store a few minutes apart. This is really frustrating, "
    "I need one of them refunded."
)

# ---- Step 1: Classify the issue -------------------------------------------------
SYS_CLASSIFY = (
    "You are a Tier-1 customer-support triage assistant for an e-commerce store. "
    "Classify tickets objectively. Do NOT invent facts not present in the ticket."
)
def prompt_classify(ticket):
    return (
        "Classify the customer ticket below.\n"
        "Return ONLY valid JSON with keys: category, subcategory, urgency "
        "(Low|Medium|High), sentiment.\n\n"
        f"TICKET:\n\"\"\"{ticket}\"\"\""
    )

# ---- Step 2: Gather the missing information --------------------------------------
SYS_GATHER = (
    "You are a support agent writing to the customer. Tone: warm, concise, professional. "
    "Ask ONLY for information needed to resolve THIS category of issue. Avoid jargon. "
    "Never ask for full card numbers, passwords, or security codes."
)
def prompt_gather(ticket, classification):
    return (
        "Given the classified ticket, decide what specific information is still missing to "
        "resolve it, then write a short message to the customer requesting exactly that.\n"
        "Return ONLY valid JSON with keys: missing_info (list of strings), "
        "customer_message (<= 60 words).\n\n"
        f"CLASSIFICATION:\n{classification}\n\nORIGINAL TICKET:\n\"\"\"{ticket}\"\"\""
    )

# ---- Step 3: Propose a solution --------------------------------------------------
SYS_SOLVE = (
    "You are a resolution specialist. Propose the smallest correct fix. Be specific and "
    "actionable. Do not promise anything outside standard policy (refunds to original "
    "payment method, 5-10 business day posting)."
)
def prompt_solve(ticket, classification, gathered):
    return (
        "Using all prior context, produce a resolution.\n"
        "Return ONLY valid JSON with keys: diagnosis, steps (list of internal actions), "
        "customer_message (<= 70 words, empathetic, no jargon).\n\n"
        f"CLASSIFICATION:\n{classification}\n\nINFO GATHERED:\n{gathered}\n\n"
        f"ORIGINAL TICKET:\n\"\"\"{ticket}\"\"\""
    )

# ---- Step 4: Escalation decision -------------------------------------------------
SYS_ESCALATE = (
    "You are a support supervisor. Decide if the case must be escalated beyond Tier 1. "
    "Escalate only for fraud, chargebacks, legal/threats, or amounts a Tier-1 agent cannot "
    "refund. Be conservative and explain briefly."
)
def prompt_escalate(classification, solution):
    return (
        "Decide whether this resolved case needs escalation.\n"
        "Return ONLY valid JSON with keys: escalate (true|false), reason, "
        "route_to, priority (P1|P2|P3).\n\n"
        f"CLASSIFICATION:\n{classification}\n\nPROPOSED RESOLUTION:\n{solution}"
    )

print("Prompts defined for all 4 steps.")


Prompts defined for all 4 steps.


## Run the chain

Notice how each call passes the **previous step's output** forward. That linkage is the
whole point of prompt chaining.


In [3]:
def show(title, text):
    print("=" * 70)
    print(title)
    print("=" * 70)
    print(text.strip(), "\n")

# Step 1
classification = call_llm(SYS_CLASSIFY, prompt_classify(TICKET), task="classify")
show("STEP 1 — CLASSIFY", classification)

# Step 2 (uses Step 1 output)
gathered = call_llm(SYS_GATHER, prompt_gather(TICKET, classification), task="gather")
show("STEP 2 — GATHER MISSING INFO", gathered)

# Step 3 (uses Steps 1 + 2)
solution = call_llm(SYS_SOLVE, prompt_solve(TICKET, classification, gathered), task="solve")
show("STEP 3 — PROPOSE SOLUTION", solution)

# Step 4 (uses Steps 1 + 3)
escalation = call_llm(SYS_ESCALATE, prompt_escalate(classification, solution), task="escalate")
show("STEP 4 — ESCALATION DECISION", escalation)


STEP 1 — CLASSIFY
{
  "category": "Billing",
  "subcategory": "Duplicate charge",
  "urgency": "High",
  "sentiment": "Frustrated"
} 

STEP 2 — GATHER MISSING INFO
{
  "missing_info": [
    "The last 4 digits of the card that was charged",
    "The date and amount of BOTH charges you see",
    "The order or invoice number tied to the purchase"
  ],
  "customer_message": "Thanks for flagging this. So we can locate the duplicate charge quickly, could you share the last 4 digits of your card, the date and amount of both charges, and the related order/invoice number?"
} 

STEP 3 — PROPOSE SOLUTION
{
  "diagnosis": "A duplicate authorization was captured for a single order.",
  "steps": [
    "Confirm both transactions reference the same order number.",
    "Void/refund the duplicate authorization to the original card.",
    "Send a refund confirmation email with the expected 5-10 day timeline."
  ],
  "customer_message": "I've confirmed a duplicate charge on your order and issued a refund 

## Assemble the final customer-facing reply

The chain's structured outputs are combined into one message the agent could send.


In [4]:
c = json.loads(classification)
g = json.loads(gathered)
s = json.loads(solution)
e = json.loads(escalation)

final_reply = (
    f"[Category: {c['category']} / {c['subcategory']} | Urgency: {c['urgency']} | "
    f"Escalate: {e['escalate']} ({e['priority']})]\n\n"
    f"{s['customer_message']}"
)
print(final_reply)


[Category: Billing / Duplicate charge | Urgency: High | Escalate: False (P3)]

I've confirmed a duplicate charge on your order and issued a refund for the extra amount to your original card. You'll get a confirmation email shortly; funds typically post back within 5-10 business days.


## Iteration / refinement (before → after)

Prompt engineering is iterative. Below is a **v1 (weak) prompt** for Step 1 and the
**v2 (constrained) prompt** we actually used. The v1 prompt is vague and produces free
text that the next step cannot parse; v2 adds a **role, an explicit field list, an enum for
urgency, a strict JSON-only constraint, and a "don't invent facts" guardrail**.


In [5]:
# v1: vague prompt (what a first draft often looks like)
SYS_V1 = "You are a helpful assistant."
PROMPT_V1 = f"Look at this support ticket and tell me about it:\n{TICKET}"

def fallback_v1(_):
    return ("This customer seems upset about being charged twice for their order this "
            "morning and wants a refund for one of the payments. It sounds like a billing "
            "problem and they'd probably like it handled quickly.")

# Simulate v1 (free-form, unstructured)
v1_out = (call_llm(SYS_V1, PROMPT_V1, task="__none__") if USE_OPENAI else fallback_v1(None))

print("----- v1 (vague prompt) output -----")
print(v1_out, "\n")
print("Problem: prose, no fields, not machine-readable -> Step 2 cannot reliably parse it.\n")

print("----- v2 (constrained prompt) output -----")
print(classification)
print("\nFix: role + explicit JSON schema + urgency enum + 'no invented facts' guardrail.")
try:
    parsed = json.loads(classification)
    print("v2 parses as JSON with keys:", list(parsed.keys()))
except Exception as ex:
    print("parse error:", ex)


----- v1 (vague prompt) output -----
This customer seems upset about being charged twice for their order this morning and wants a refund for one of the payments. It sounds like a billing problem and they'd probably like it handled quickly. 

Problem: prose, no fields, not machine-readable -> Step 2 cannot reliably parse it.

----- v2 (constrained prompt) output -----
{
  "category": "Billing",
  "subcategory": "Duplicate charge",
  "urgency": "High",
  "sentiment": "Frustrated"
}

Fix: role + explicit JSON schema + urgency enum + 'no invented facts' guardrail.
v2 parses as JSON with keys: ['category', 'subcategory', 'urgency', 'sentiment']


## Summary

- **What each step does & how it uses prior output:**
  1. **Classify** turns raw ticket text into structured fields (category/urgency/sentiment).
  2. **Gather** reads the *category* to ask only for the relevant missing info.
  3. **Solve** uses the classification + gathered info to produce a diagnosis + reply.
  4. **Escalate** reads the classification + proposed solution to make a routing decision.
- **Iteration shown:** v1 (vague) → v2 (role + JSON schema + constraints), which is what
  makes the chain robust because every step can parse the previous step's output.
- **Constraints applied throughout:** role per step, tone, field/enum requirements, length
  limits, JSON-only output, and safety guardrails (never ask for full card numbers).
